In [ ]:
import pandas as pd

df = pd.read_csv(r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\caresuper.csv", encoding="cp1252")
df.head()

,Table 1 – Assets,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,NaN,NaN,NaN,NaN,NaN
1,Portfolio Holdings Information for Investment ...,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,Value (AUD),Weighting (%)
3,Cash,NaN,NaN,NaN,NaN
4,Name of Institution,Currency,NaN,Value (AUD),Weighting (%)


In [ ]:
import pandas as pd

file_path = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\caresuper.csv"

# Keep only Table 1 (stop at "Table 2")
lines = []
with open(file_path, "r", encoding="cp1252") as f:
    for line in f:
        if "Table 2" in line:
            break
        lines.append(line)

# Overwrite with only Table 1
with open(file_path, "w", encoding="cp1252") as f:
    f.writelines(lines)

# Load cleaned file
df = pd.read_csv(file_path, encoding="cp1252")
df.head()


,Table 1 – Assets,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,NaN,NaN,NaN,NaN,NaN
1,Portfolio Holdings Information for Investment ...,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,Value (AUD),Weighting (%)
3,Cash,NaN,NaN,NaN,NaN
4,Name of Institution,Currency,NaN,Value (AUD),Weighting (%)


In [5]:
import pandas as pd
import csv
import re

# File paths
raw_path = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\caresuper.csv"
cleaned_path = r"D:\LinhDao\Programming\SUPERFUNdProject\CareSuper_Cleaned.csv"
lookup_path = r"D:\LinhDao\Programming\SUPERFUNdProject\InternationalCountryCodes.csv"

# Target schema
columns = [
    "Effective Date",
    "Fund Name",
    "Option Name",
    "Asset Class Name",
    "Int/Ext",
    "Name/Kind of Investment Item",
    "Currency",
    "Stock ID",
    "Listed Country",
    "Units Held",
    "% Ownership",
    "Address",
    "Value (AUD)",
    "Weighting"
]

all_rows = []

def make_base_row(asset_class_name, int_ext):
    """Create a blank row with common defaults."""
    row = {col: "" for col in columns}
    row.update({
        "Effective Date": "31/12/2024",
        "Fund Name": "Care Super",
        "Option Name": "Balanced",
        "Asset Class Name": asset_class_name,
        "Int/Ext": int_ext,
    })
    return row

def clean_cell(target_col: str, val: str):
    """Per-cell cleaning logic applied to all rows (normal + subtotal)."""
    if val is None:
        return ""
    val = str(val).strip()

    if target_col in ["Weighting"] and val:
        if val.endswith("%"):
            try:
                return float(val.replace("%", "")) / 100
            except:
                return val  # leave as-is if unparsable
        # keep as-is (no forced float) if no % sign, per your preference
        return val

    if target_col == "Value (AUD)":
        # allow digits, commas, dots, minus; blank it if non-numeric
        if not re.match(r'^[0-9,.\-]+$', val):
            return ""
        return val

    return val

with open(raw_path, "r", encoding="cp1252") as f:
    reader = csv.reader(f)
    rows = list(reader)

i = 0
while i < len(rows):
    row = rows[i]

    # Detect sub-table header
    if any("Name" in str(cell) for cell in row) and \
       any("Value" in str(cell) for cell in row) and \
       any("Weighting" in str(cell) for cell in row):

        # Row above header has Asset Class + Int/Ext
        if i > 0:
            meta_cell = rows[i-1][0]
            meta_parts = [line.strip() for line in meta_cell.splitlines() if line.strip()]

            # Asset Class Name = first two words of first line
            first_line = meta_parts[0]
            asset_class_name = " ".join(first_line.split()[:2])

            # Int/Ext = last line
            last_line = meta_parts[-1].lower()
            if "internal" in last_line:
                int_ext = 0
            elif "external" in last_line:
                int_ext = 1
            else:
                int_ext = ""

        # Map columns dynamically
        header_map = {}
        for j, col in enumerate(row):
            col = str(col).strip()
            if not col:
                continue
            if col.startswith("Name"):
                header_map[j] = "Name/Kind of Investment Item"
            elif "Value" in col:
                header_map[j] = "Value (AUD)"
            elif "Value (AUD)" in col:
                header_map[j] = "Value (AUD)"    
            elif "Weighting" in col:
                header_map[j] = "Weighting"
            elif "Weighting (%)" in col:
                header_map[j] = "Weighting"
            elif "% of property held" in col:
                header_map[j] = "% Ownership"
            elif "% of property held" in col:
                header_map[j] = "% Ownership"    
            elif "Units held" in col:
                header_map[j] = "Units Held"    
            elif "Security Identifier" in col:
                header_map[j] = "Stock ID"  # map Security Identifier → Stock ID column
            else:
                for target in columns:
                    if col.lower().startswith(target.lower()):
                        header_map[j] = target

        # Collect rows until (and including) the "Total" row
        j = i + 1
        while j < len(rows):
            data_row = rows[j]

            # Build row (same for normal and subtotal)
            new_row = make_base_row(asset_class_name, int_ext)

            for idx, target_col in header_map.items():
                if idx < len(data_row):
                    new_row[target_col] = clean_cell(target_col, data_row[idx])

            # Post-process Stock ID & Listed Country (same for all rows)
            if new_row["Stock ID"]:
                sid = str(new_row["Stock ID"])
                if len(sid) > 2:
                    new_row["Listed Country"] = sid[-2:]
                    new_row["Stock ID"] = sid[:-2]

            # If this source row is the "Total" row, just rename Name and stop the block
            if any(str(cell).strip() == "Total" for cell in data_row):
                new_row["Name/Kind of Investment Item"] = "Sub Total"
                all_rows.append(new_row)
                break

            # Normal row
            all_rows.append(new_row)
            j += 1

        i = j
    i += 1

# Save to cleaned file
df_cleaned = pd.DataFrame(all_rows, columns=columns)

# === Country lookup (first col = Country, second = Code) ===
try:
    lu = pd.read_csv(lookup_path, encoding="cp1252", usecols=[0, 1])
    lu.columns = ["Country", "Code"]  # enforce names
    code2country = dict(zip(lu["Code"].astype(str).str.strip().str.upper(),
                            lu["Country"].astype(str).str.strip()))
    df_cleaned["Listed Country"] = df_cleaned["Listed Country"].apply(
        lambda x: code2country.get(str(x).strip().upper(), x if pd.notna(x) else "")
    )
except Exception as e:
    print(f"⚠️ Country lookup skipped due to error: {e}")

# Fill Int/Ext with 1 if empty and ensure integers
df_cleaned["Int/Ext"] = df_cleaned["Int/Ext"].replace("", pd.NA).fillna(1).astype(int)

df_cleaned.to_csv(cleaned_path, index=False, encoding="cp1252")

print(f"✅ Cleaning complete! Saved to {cleaned_path}")


✅ Cleaning complete! Saved to D:\LinhDao\Programming\SUPERFUNdProject\CareSuper_Cleaned.csv


C:\Users\thuon\AppData\Local\Temp\ipykernel_25776\3310458741.py:173: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cleaned["Int/Ext"] = df_cleaned["Int/Ext"].replace("", pd.NA).fillna(1).astype(int)


In [3]:
import chardet

with open(r"D:\LinhDao\Programming\SUPERFUNdProject\caresuper.csv", "rb") as f:
    result = chardet.detect(f.read(100000))  # read first 100 KB
print(result)

{'encoding': 'Windows-1252', 'confidence': 0.73, 'language': ''}


In [ ]:
#new version - Aug 20
import pandas as pd
import csv
import re
import numpy as np

# File paths
raw_path = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\caresuper.csv"
cleaned_path = r"D:\LinhDao\Programming\SUPERFUNdProject\CareSuper_Cleaned_final.csv"

# Target schema (column order)
columns = [
    "Effective Date",
    "Fund Name",
    "Option Name",
    "Asset Class Name",
    "Int/Ext",
    "Name/Kind of Investment Item",
    "Currency",
    "Stock ID",
    "Listed Country",
    "Units Held",
    "% Ownership",
    "Address",
    "Value (AUD)",
    "Weighting"
]

all_rows = []

def make_base_row(asset_class_name, int_ext):
    """Create a blank row with common defaults."""
    row = {col: "" for col in columns}
    row.update({
        "Effective Date": "31/12/2024",
        "Fund Name": "Care Super",
        "Option Name": "Balanced",
        "Asset Class Name": asset_class_name,
        "Int/Ext": int_ext,
    })
    return row

# ---------- Robust numeric parsers (preserve blanks) -----------------
def _normalize_signs_spaces(s: str) -> str:
    # Remove normal & non-breaking spaces; normalize minus/dashes
    return (
        s.replace("\u00A0", "")   # non-breaking space
         .replace("\u2212", "-")  # minus sign
         .replace("−", "-")       # another minus
         .replace("–", "-")       # en dash
         .replace("—", "-")       # em dash
         .strip()
    )

def _to_units_float(val):
    """Units Held: commas allowed, parentheses negatives allowed, spaces stripped."""
    if val is None:
        return ""
    s = _normalize_signs_spaces(str(val))
    if s == "":
        return ""
    if s.startswith("(") and s.endswith(")"):  # parentheses negative
        s = "-" + s[1:-1].strip()
    s = s.replace(",", "")  # remove thousands separators
    try:
        return float(s)
    except ValueError:
        return ""

def _to_value_aud_float(val):
    """Value (AUD): like Units Held, plus strip leading $ if present."""
    if val is None:
        return ""
    s = _normalize_signs_spaces(str(val))
    if s == "":
        return ""
    if s.startswith("$"):  # strip leading $
        s = s[1:].strip()
    if s.startswith("(") and s.endswith(")"):  # parentheses negative
        s = "-" + s[1:-1].strip()
    s = s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return ""
# --------------------------------------------------------------------

def clean_cell(target_col: str, val: str):
    """Per-cell cleaning logic applied to all rows (normal + subtotal)."""
    if val is None:
        return ""
    val = str(val).strip()

    if target_col in ["Weighting"] and val:
        if val.endswith("%"):
            try:
                return float(val.replace("%", "")) / 100
            except:
                return val
        return val

    if target_col == "Units Held":
        return _to_units_float(val)

    if target_col == "Value (AUD)":
        return _to_value_aud_float(val)

    if target_col == "Stock ID":  # force Stock ID always as string
        return str(val).strip()

    return val

# -------------------- Read and parse sub-tables ----------------------
with open(raw_path, "r", encoding="cp1252") as f:
    reader = csv.reader(f)
    rows = list(reader)

i = 0
while i < len(rows):
    row = rows[i]

    if any("Name" in str(cell) for cell in row) and \
       any("Value" in str(cell) for cell in row) and \
       any("Weighting" in str(cell) for cell in row):

        if i > 0:
            meta_cell = rows[i-1][0]
            meta_parts = [line.strip() for line in meta_cell.splitlines() if line.strip()]
            first_line = meta_parts[0]
            asset_class_name = " ".join(first_line.split()[:2])
            last_line = meta_parts[-1].lower()
            if "internal" in last_line:
                int_ext = 0
            elif "external" in last_line:
                int_ext = 1
            else:
                int_ext = ""

        header_map = {}
        for j, col in enumerate(row):
            col = str(col).strip()
            if not col:
                continue
            if col.startswith("Name"):
                header_map[j] = "Name/Kind of Investment Item"
            elif "Value (AUD)" in col or "Value" in col:
                header_map[j] = "Value (AUD)"
            elif "Weighting (%)" in col or "Weighting" in col:
                header_map[j] = "Weighting"
            elif "% of property held" in col:
                header_map[j] = "% Ownership"
            elif "Units held" in col:
                header_map[j] = "Units Held"
            elif "Security Identifier" in col or "Stock ID" in col:
                header_map[j] = "Stock ID"
            else:
                for target in columns:
                    if col.lower().startswith(target.lower()):
                        header_map[j] = target

        j = i + 1
        while j < len(rows):
            data_row = rows[j]
            new_row = make_base_row(asset_class_name, int_ext)

            for idx, target_col in header_map.items():
                if idx < len(data_row):
                    new_row[target_col] = clean_cell(target_col, data_row[idx])

            # Stock ID: keep as string; trim last 2 chars -> Listed Country
            if new_row["Stock ID"]:
                sid = str(new_row["Stock ID"])
                if len(sid) > 2:
                    new_row["Listed Country"] = sid[-2:]
                    new_row["Stock ID"] = sid[:-2]

            if any(str(cell).strip() == "Total" for cell in data_row):
                new_row["Name/Kind of Investment Item"] = "Sub Total"
                all_rows.append(new_row)
                break

            all_rows.append(new_row)
            j += 1

        i = j
    i += 1

df_cleaned = pd.DataFrame(all_rows, columns=columns)

# Ensure key/text fields are strings
df_cleaned["Stock ID"] = df_cleaned["Stock ID"].astype("string")
df_cleaned["Listed Country"] = df_cleaned["Listed Country"].astype("string")

# Fill Int/Ext with 1 if empty and ensure integers
df_cleaned["Int/Ext"] = df_cleaned["Int/Ext"].replace("", pd.NA).fillna(1).astype(int)

# Ensure numeric columns are floats in final DataFrame
df_cleaned["Units Held"] = pd.to_numeric(df_cleaned["Units Held"], errors="coerce")
df_cleaned["Value (AUD)"] = pd.to_numeric(df_cleaned["Value (AUD)"], errors="coerce")

df_cleaned.to_csv(cleaned_path, index=False, encoding="cp1252")
print(f"✅ Cleaning complete! Saved to {cleaned_path}")


[RAW]   Units Held=None | Value (AUD)='110,893,396'
[PARSED] Units Held= | Value (AUD)=110893396.0
[RAW]   Units Held=None | Value (AUD)='2,160,863'
[PARSED] Units Held= | Value (AUD)=2160863.0
[RAW]   Units Held=None | Value (AUD)='-860,394'
[PARSED] Units Held= | Value (AUD)=-860394.0
[RAW]   Units Held=None | Value (AUD)='351,428'
[PARSED] Units Held= | Value (AUD)=351428.0
[RAW]   Units Held=None | Value (AUD)='140,846,685'
[PARSED] Units Held= | Value (AUD)=140846685.0
[RAW]   Units Held=None | Value (AUD)='127,249,299'
[PARSED] Units Held= | Value (AUD)=127249299.0
[RAW]   Units Held=None | Value (AUD)='107,590'
[PARSED] Units Held= | Value (AUD)=107590.0
[RAW]   Units Held=None | Value (AUD)='-76,536'
[PARSED] Units Held= | Value (AUD)=-76536.0
[RAW]   Units Held=None | Value (AUD)='161,325,379'
[PARSED] Units Held= | Value (AUD)=161325379.0
[RAW]   Units Held=None | Value (AUD)='678,107,650'
[PARSED] Units Held= | Value (AUD)=678107650.0
[RAW]   Units Held=None | Value (AUD)='2

C:\Users\thuon\AppData\Local\Temp\ipykernel_25776\1525841473.py:243: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cleaned["Int/Ext"] = df_cleaned["Int/Ext"].replace("", pd.NA).fillna(1).astype(int)
